# ⚡ EV Charging Infrastructure Analytics Dashboard

**Project:** EV Charging Demand, Utilization & Infrastructure Analytics  
**Dataset:** 5,000 stations × 17 columns (synthetic/generated data)  
**Phase 10:** Interactive Dashboard using Python + Plotly

> ⚠️ This dataset is synthetic. All relationships are **associative**, not causal.  
> Near-perfect ML performance reflects data generation, not production readiness.

---

In [18]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
import warnings
warnings.filterwarnings('ignore')

# Plotly defaults
import plotly.io as pio

pio.templates.default = "plotly_dark"

COLORS = ["#6C63FF", "#00D4AA", "#FF6B6B", "#FFD93D", "#45B7D1",
          "#F78FB3", "#778BEB", "#E77F67", "#55E6C1", "#C7ECEE"]

print("Libraries loaded successfully")

Libraries loaded successfully


## 📂 Data Loading & Enrichment

In [19]:
# Load the processed station-level dataset
df = pd.read_csv("../../data/processed/station_features.csv")

print(f"Dataset: {df.shape[0]:,} stations × {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")

Dataset: 5,000 stations × 17 columns
Columns: ['Station_ID', 'Number_of_Chargers', 'Max_Station_Power_kW', 'Parking_Spots', 'Station_Age_Years', 'Charger_Type', 'Station_Type', 'Renewable_Energy_Source', 'Avg_Utilization', 'Congestion_Freq', 'Total_Sessions', 'Avg_Wait_Time', 'Avg_Session_Duration', 'Power_per_Charger', 'Charger_to_Parking_Ratio', 'Target_High_Util', 'Target_High_Congestion']


In [33]:
# ── Computed columns (in-memory only, raw data untouched) ──

# Energy delivered per station (kWh)
df["Energy_kWh"] = df["Total_Sessions"] * (df["Avg_Session_Duration"] / 60) * df["Power_per_Charger"]

# Revenue at $0.305 / kWh
df["Revenue"] = df["Energy_kWh"] * 0.305

# Gap Score — decision-support indicator (NOT an ROI metric)
wait_max = df["Avg_Wait_Time"].max()
df["Gap_Score"] = (
    df["Avg_Utilization"] * 0.4
    + df["Congestion_Freq"] * 0.4
    + (df["Avg_Wait_Time"] / wait_max) * 0.2
)

# Segment assignment
def assign_segment(row):
    u, c = row["Avg_Utilization"], row["Congestion_Freq"]
    if u >= 0.65 and c >= 0.50:
        return "Expansion Candidates"
    elif u >= 0.45 and c < 0.50:
        return "Healthy Giants"
    elif u < 0.25 and c < 0.10:
        return "Overbuilt"
    elif u < 0.25 and c >= 0.10:
        return "Sleepy"
    else:
        return "Monitor"

df["Segment"] = df.apply(assign_segment, axis=1)

# Recommended Action
action_map = {
    "Expansion Candidates": "EXPAND",
    "Healthy Giants": "MONITOR",
    "Overbuilt": "AVOID FURTHER CAPEX",
    "Monitor": "OPTIMIZE",
    "Sleepy": "OPTIMIZE",
}
df["Recommended_Action"] = df["Segment"].map(action_map)

# Priority
df["Priority"] = pd.cut(
    df["Gap_Score"],
    bins=[0, 0.30, 0.55, 1.0],
    labels=["Low", "Medium", "High"],
    include_lowest=True,
)

print(f" Enrichment complete — {df.shape[1]} columns now available")
print(f"   Segments: {df['Segment'].value_counts().to_dict()}")

 Enrichment complete — 23 columns now available
   Segments: {'Monitor': 1421, 'Overbuilt': 1327, 'Expansion Candidates': 917, 'Healthy Giants': 671, 'Sleepy': 664}


---
# 📊 Page 1 — Executive Overview
Network-level KPIs and high-congestion stations

In [21]:
# ── KPI Cards ──

total_sessions = df["Total_Sessions"].sum()
energy = df["Energy_kWh"].sum()
revenue = df["Revenue"].sum()
avg_util = df["Avg_Utilization"].mean() * 100
avg_cong = df["Congestion_Freq"].mean() * 100
avg_wait = df["Avg_Wait_Time"].mean()

kpi_style = '''
<style>
.kpi-row { display: flex; gap: 16px; margin-bottom: 16px; }
.kpi-box {
    flex: 1; padding: 22px 16px; text-align: center;
    background: linear-gradient(135deg, #1a1a2e, #2a2a3c);
    border: 1px solid rgba(108, 99, 255, 0.2);
    border-radius: 14px;
    font-family: 'Segoe UI', sans-serif;
}
.kpi-label { color: #888; font-size: 12px; text-transform: uppercase; letter-spacing: 1.2px; margin-bottom: 4px; }
.kpi-val { font-size: 28px; font-weight: 700; background: linear-gradient(90deg, #6C63FF, #00D4AA);
           -webkit-background-clip: text; -webkit-text-fill-color: transparent; }
</style>
'''

html = kpi_style + '''
<div class="kpi-row">
  <div class="kpi-box"><div class="kpi-label">Total Sessions</div><div class="kpi-val">''' + f"{total_sessions:,}" + '''</div></div>
  <div class="kpi-box"><div class="kpi-label">Energy Delivered</div><div class="kpi-val">''' + f"{energy/1e6:.2f}M kWh" + '''</div></div>
  <div class="kpi-box"><div class="kpi-label">Revenue</div><div class="kpi-val">''' + f"${revenue/1e6:.2f}M" + '''</div></div>
</div>
<div class="kpi-row">
  <div class="kpi-box"><div class="kpi-label">Avg Utilization</div><div class="kpi-val">''' + f"{avg_util:.1f}%" + '''</div></div>
  <div class="kpi-box"><div class="kpi-label">Congestion Rate</div><div class="kpi-val">''' + f"{avg_cong:.1f}%" + '''</div></div>
  <div class="kpi-box"><div class="kpi-label">Avg Wait Time</div><div class="kpi-val">''' + f"{avg_wait:.2f} min" + '''</div></div>
</div>
'''

display(HTML(html))

In [22]:
# ── Top 10 Stations by Congestion ──

top10_cong = (
    df.nlargest(10, "Congestion_Freq")
    [["Station_ID", "Congestion_Freq", "Avg_Wait_Time", "Avg_Utilization", "Number_of_Chargers"]]
    .reset_index(drop=True)
)

fig = go.Figure(go.Table(
    header=dict(
        values=["<b>Station ID</b>", "<b>Congestion</b>", "<b>Wait (min)</b>",
                "<b>Utilization</b>", "<b>Chargers</b>"],
        fill_color="#1a1a2e", font=dict(color="white", size=13),
        align="center", line_color="#333"
    ),
    cells=dict(
        values=[
            top10_cong["Station_ID"],
            (top10_cong["Congestion_Freq"] * 100).round(1).astype(str) + "%",
            top10_cong["Avg_Wait_Time"].round(2),
            (top10_cong["Avg_Utilization"] * 100).round(1).astype(str) + "%",
            top10_cong["Number_of_Chargers"],
        ],
        fill_color=[["#2a2a3c", "#1e1e2e"] * 5],
        font=dict(color="white", size=12),
        align="center", line_color="#333", height=30,
    )
))
fig.update_layout(
    title="Top 10 Stations by Congestion Frequency",
    height=420, margin=dict(l=20, r=20, t=50, b=20),
    paper_bgcolor="rgba(0,0,0,0)",
)
fig.show()

---
# 🏢 Page 2 — Station Performance
Top demand stations and performance by station/charger type

In [23]:
# ── Top 10 Stations by Charging Demand ──

top10_dem = (
    df.nlargest(10, "Total_Sessions")
    [["Station_ID", "Total_Sessions", "Avg_Utilization", "Avg_Wait_Time",
      "Congestion_Freq", "Number_of_Chargers", "Target_High_Util", "Target_High_Congestion"]]
    .reset_index(drop=True)
)

fig = go.Figure(go.Table(
    header=dict(
        values=["<b>Station</b>", "<b>Sessions</b>", "<b>Utilization</b>",
                "<b>Wait (min)</b>", "<b>Congestion</b>", "<b>Chargers</b>",
                "<b>High Util?</b>", "<b>High Cong?</b>"],
        fill_color="#1a1a2e", font=dict(color="white", size=12),
        align="center", line_color="#333"
    ),
    cells=dict(
        values=[
            top10_dem["Station_ID"],
            top10_dem["Total_Sessions"],
            (top10_dem["Avg_Utilization"] * 100).round(1).astype(str) + "%",
            top10_dem["Avg_Wait_Time"].round(2),
            (top10_dem["Congestion_Freq"] * 100).round(1).astype(str) + "%",
            top10_dem["Number_of_Chargers"],
            top10_dem["Target_High_Util"].map({1: "✅", 0: "❌"}),
            top10_dem["Target_High_Congestion"].map({1: "🔴", 0: "🟢"}),
        ],
        fill_color=[["#2a2a3c", "#1e1e2e"] * 5],
        font=dict(color="white", size=12),
        align="center", line_color="#333", height=30,
    )
))
fig.update_layout(
    title="Top 10 Stations by Charging Demand (Total Sessions)",
    height=420, margin=dict(l=20, r=20, t=50, b=20),
    paper_bgcolor="rgba(0,0,0,0)",
)
fig.show()

In [24]:
# ── Charging Demand by Station Type ──

grp_st = (
    df.groupby("Station_Type", as_index=False)["Total_Sessions"]
    .sum()
    .sort_values("Total_Sessions", ascending=True)
)

fig = px.bar(
    grp_st, y="Station_Type", x="Total_Sessions",
    orientation="h", color="Station_Type",
    color_discrete_sequence=COLORS,
    text="Total_Sessions",
    title="Charging Demand by Station Type"
)
fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig.update_layout(showlegend=False, height=350,
                  xaxis_title="Total Sessions", yaxis_title="")
fig.show()

In [25]:
# ── Avg Utilization by Charger Type ──

grp_ct = (
    df.groupby("Charger_Type", as_index=False)["Avg_Utilization"]
    .mean()
    .sort_values("Avg_Utilization", ascending=True)
)
grp_ct["Util_Pct"] = grp_ct["Avg_Utilization"] * 100

fig = px.bar(
    grp_ct, y="Charger_Type", x="Util_Pct",
    orientation="h", color="Charger_Type",
    color_discrete_sequence=COLORS[2:],
    text="Util_Pct",
    title="Average Utilization by Charger Type"
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(showlegend=False, height=350,
                  xaxis_title="Avg Utilization (%)", yaxis_title="")
fig.show()

---
# 🔍 Page 3 — Root Cause Analysis
Exploring infrastructure characteristics **associated with** utilization & congestion

> All relationships shown are **associative**, not causal.

In [26]:
# ── Correlation Heatmap — Key Metrics ──

corr_cols = [
    "Number_of_Chargers", "Max_Station_Power_kW", "Parking_Spots",
    "Station_Age_Years", "Avg_Utilization", "Congestion_Freq",
    "Total_Sessions", "Avg_Wait_Time", "Avg_Session_Duration",
    "Power_per_Charger", "Charger_to_Parking_Ratio",
]
corr = df[corr_cols].corr().round(2)

# Shorter labels for readability
short = {
    "Number_of_Chargers": "Chargers", "Max_Station_Power_kW": "Max Power",
    "Parking_Spots": "Parking", "Station_Age_Years": "Age",
    "Avg_Utilization": "Utilization", "Congestion_Freq": "Congestion",
    "Total_Sessions": "Sessions", "Avg_Wait_Time": "Wait Time",
    "Avg_Session_Duration": "Duration", "Power_per_Charger": "Pwr/Charger",
    "Charger_to_Parking_Ratio": "Chgr/Park Ratio",
}
corr.index = [short[c] for c in corr.index]
corr.columns = [short[c] for c in corr.columns]

fig = px.imshow(
    corr, text_auto=True, color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1, aspect="auto",
    title="Correlation Heatmap — Key Infrastructure & Performance Metrics"
)
fig.update_layout(height=580, xaxis_tickangle=-45,
                  coloraxis_colorbar_title="r")
fig.show()

In [27]:
# ── Utilization & Congestion by Number of Chargers (grouped bar) ──

grp_ch = (
    df.groupby("Number_of_Chargers", as_index=False)
    .agg(Avg_Util=("Avg_Utilization", "mean"),
         Avg_Cong=("Congestion_Freq", "mean"))
)
grp_ch["Avg_Util"] = (grp_ch["Avg_Util"] * 100).round(1)
grp_ch["Avg_Cong"] = (grp_ch["Avg_Cong"] * 100).round(1)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=grp_ch["Number_of_Chargers"], y=grp_ch["Avg_Util"],
    name="Avg Utilization %", marker_color="#6C63FF",
    text=grp_ch["Avg_Util"], textposition="outside",
))
fig.add_trace(go.Bar(
    x=grp_ch["Number_of_Chargers"], y=grp_ch["Avg_Cong"],
    name="Avg Congestion %", marker_color="#FF6B6B",
    text=grp_ch["Avg_Cong"], textposition="outside",
))
fig.update_layout(
    barmode="group", height=420,
    title="Utilization & Congestion Associated with Charger Count",
    xaxis_title="Number of Chargers", yaxis_title="Percentage (%)",
    legend=dict(orientation="h", y=-0.15, x=0.5, xanchor="center"),
)
fig.show()

In [28]:
# ── Utilization Distribution by Station Type (box plot) ──

df_box = df.copy()
df_box["Util_Pct"] = df_box["Avg_Utilization"] * 100

fig = px.box(
    df_box, x="Station_Type", y="Util_Pct",
    color="Station_Type", color_discrete_sequence=COLORS,
    title="Utilization Distribution by Station Type"
)
fig.update_layout(showlegend=False, height=420,
                  yaxis_title="Utilization (%)", xaxis_title="")
fig.show()

---
# 🛠️ Page 4 — Infrastructure Action Plan
Decision-support segmentation and gap-score prioritization

> **Gap Score** is an analytical prioritization indicator — NOT a guaranteed ROI or investment metric.

In [29]:
# ── Station Count by Segment ──

seg_order = ["Expansion Candidates", "Healthy Giants", "Monitor", "Sleepy", "Overbuilt"]
seg_colors = {
    "Expansion Candidates": "#FF6B6B", "Healthy Giants": "#00D4AA",
    "Monitor": "#FFD93D", "Sleepy": "#778BEB", "Overbuilt": "#45B7D1",
}

seg_counts = (
    df["Segment"].value_counts()
    .reindex(seg_order, fill_value=0)
    .reset_index()
)
seg_counts.columns = ["Segment", "Count"]

fig = px.bar(
    seg_counts, x="Segment", y="Count", color="Segment",
    color_discrete_map=seg_colors, text="Count",
    title="Station Count by Infrastructure Segment"
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False, height=420, xaxis_tickangle=-15)
fig.show()

In [30]:
# ── Avg Gap Score by Station Type ──

grp_gap = (
    df.groupby("Station_Type", as_index=False)["Gap_Score"]
    .mean()
    .sort_values("Gap_Score", ascending=True)
)

fig = px.bar(
    grp_gap, y="Station_Type", x="Gap_Score",
    orientation="h", color="Station_Type",
    color_discrete_sequence=COLORS, text="Gap_Score",
    title="Avg Gap Score by Station Type"
)
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(showlegend=False, height=350, xaxis_title="Gap Score", yaxis_title="")
fig.show()

In [31]:
# ── Top 15 Priority Stations (by Gap Score) ──

top15 = (
    df.nlargest(15, "Gap_Score")
    [["Station_ID", "Gap_Score", "Segment", "Recommended_Action", "Priority",
      "Avg_Utilization", "Congestion_Freq", "Avg_Wait_Time",
      "Number_of_Chargers", "Station_Type"]]
    .reset_index(drop=True)
)

fig = go.Figure(go.Table(
    header=dict(
        values=["<b>" + c.replace("_", " ") + "</b>" for c in top15.columns],
        fill_color="#1a1a2e", font=dict(color="white", size=11),
        align="center", line_color="#333"
    ),
    cells=dict(
        values=[
            top15["Station_ID"],
            top15["Gap_Score"].round(3),
            top15["Segment"],
            top15["Recommended_Action"],
            top15["Priority"].astype(str),
            (top15["Avg_Utilization"] * 100).round(1).astype(str) + "%",
            (top15["Congestion_Freq"] * 100).round(1).astype(str) + "%",
            top15["Avg_Wait_Time"].round(2),
            top15["Number_of_Chargers"],
            top15["Station_Type"],
        ],
        fill_color=[["#2a2a3c", "#1e1e2e"] * 8],
        font=dict(color="white", size=11),
        align="center", line_color="#333", height=28,
    )
))
fig.update_layout(
    title="Top 15 Priority Stations — Gap Score Ranking",
    height=560, margin=dict(l=20, r=20, t=50, b=20),
    paper_bgcolor="rgba(0,0,0,0)",
)
fig.show()

In [32]:
# ── Segment Distribution (donut chart) ──

fig = px.pie(
    seg_counts, names="Segment", values="Count",
    color="Segment", color_discrete_map=seg_colors,
    hole=0.45, title="Infrastructure Segment Distribution"
)
fig.update_traces(textinfo="percent+label", textfont_size=12)
fig.update_layout(height=420,
    legend=dict(orientation="h", y=-0.1, x=0.5, xanchor="center"))
fig.show()

---
### 📌 Key limitations


| Item | Note |
|------|------|
| **Data** | Synthetic/generated — not real operational measurements |
| **Relationships** | Associative only — no causal claims |
| **ML Performance** | Near-perfect results reflect synthetic data generation |
| **Gap Score** | Decision-support indicator, not investment optimization |
| **Temporal Patterns** | Demand was unusually flat — no rush-hour peaks found |
| **Weather/Traffic** | Weak relationships — not overstated |

---
*EV Charging Infrastructure Analytics — Phase 10 Dashboard*